# Chapter 4: Probability & Statistics
## Notebook 03 - Advanced

Hypothesis testing, confidence intervals, A/B testing for ML experiments, and capstone analysis.

**What you'll learn:**
- Z-test and t-test with real examples
- Confidence intervals: compute and visualize
- A/B testing for model comparison
- Correlation vs causation
- P-values: meaning and misconceptions

**Time estimate:** 2.5 hours

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*

## 1. Hypothesis Testing Workflow

See `assets/diagrams/hypothesis_testing.svg` for the flowchart.

1. Formulate H₀ (null) and H₁ (alternative)
2. Choose α (significance level, e.g., 0.05)
3. Collect data
4. Compute test statistic (z, t, etc.)
5. Compare to critical value or p-value
6. Decision: Reject H₀ or fail to reject

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Z-test: known population variance. Compare sample mean to known μ₀.
# Example: Model A has historical accuracy μ₀=0.85. New run: n=100, mean=0.88, σ=0.1
n = 100
x_bar = 0.88
sigma = 0.1  # population std known
mu_0 = 0.85

z = (x_bar - mu_0) / (sigma / np.sqrt(n))
p_value = 2 * (1 - stats.norm.cdf(abs(z)))  # two-tailed

print("Z-test: Is new model accuracy significantly different from 0.85?")
print(f"  z = {z:.3f}")
print(f"  p-value = {p_value:.4f}")
print(f"  Reject H₀ at α=0.05? {p_value < 0.05}")

In [ ]:
# T-test: unknown variance. Compare two groups.
# Example: Model A vs Model B accuracy on same test set
model_a_scores = np.random.normal(0.87, 0.03, 50)
model_b_scores = np.random.normal(0.85, 0.04, 50)

t_stat, p_value = stats.ttest_ind(model_a_scores, model_b_scores)
print("Independent t-test: Model A vs Model B")
print(f"  t = {t_stat:.3f}")
print(f"  p-value = {p_value:.4f}")
print(f"  Reject H₀ (no difference)? {p_value < 0.05}")

## 2. Confidence Intervals

95% CI for mean: $\bar{x} \pm 1.96 \frac{\sigma}{\sqrt{n}}$ (z) or use t-distribution when σ unknown.

In [ ]:
def confidence_interval(sample, confidence=0.95):
    """Compute (low, high) for mean using t-distribution."""
    n = len(sample)
    mean = np.mean(sample)
    sem = stats.sem(sample)
    h = sem * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean - h, mean + h

data = np.random.normal(0.88, 0.05, 100)
low, high = confidence_interval(data)
print(f"95% CI for mean: [{low:.4f}, {high:.4f}]")
print(f"True mean in interval? {0.88 >= low and 0.88 <= high} (we know it's 0.88)")

In [ ]:
# Visualize many CIs: 20 samples, each with 95% CI
np.random.seed(42)
true_mean = 0.85
cis = []
for _ in range(20):
    sample = np.random.normal(true_mean, 0.1, 50)
    low, high = confidence_interval(sample)
    cis.append((low, high))

fig, ax = plt.subplots(figsize=(8, 6))
for i, (lo, hi) in enumerate(cis):
    ax.plot([lo, hi], [i, i], 'b-o', markersize=4)
ax.axvline(x=true_mean, color='red', linestyle='--', label='True mean')
ax.set_xlabel('Mean')
ax.set_ylabel('Sample index')
ax.set_title('95% Confidence Intervals (most should contain true mean)')
ax.legend()
plt.tight_layout()
plt.savefig('../assets/diagrams/confidence_intervals.svg')
plt.show()

## 3. A/B Testing for ML Models

Compare conversion rates (or accuracy as proportion) between variant A and B. Use two-proportion z-test.

In [ ]:
def ab_test_two_proportion(n_a, conv_a, n_b, conv_b):
    """Two-proportion z-test for A/B test. Returns (z, p_value)."""
    p_a = conv_a / n_a
    p_b = conv_b / n_b
    p_pool = (conv_a + conv_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    if se == 0:
        return 0.0, 1.0
    z = (p_a - p_b) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_value

# Example: Model A: 120 conversions of 500, Model B: 100 conversions of 500
z, p = ab_test_two_proportion(500, 120, 500, 100)
print("A/B Test: Model A (24% conv) vs Model B (20% conv)")
print(f"  z = {z:.3f}, p = {p:.4f}")
print(f"  Significant at α=0.05? {p < 0.05}")

## 4. Correlation vs Causation

**Correlation ≠ Causation**. Ice cream sales and drownings are correlated (summer) but one doesn't cause the other.

In ML: feature X correlated with label Y doesn't mean X causes Y—could be confounding.

In [ ]:
# Spurious correlation: X and Y both depend on Z
np.random.seed(42)
z = np.random.rand(100)  # confounder
x = z + np.random.randn(100) * 0.2
y = 2 * z + np.random.randn(100) * 0.2

r, p = stats.pearsonr(x, y)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, alpha=0.7)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'Correlation r={r:.3f} (spurious: both depend on Z)')
plt.tight_layout()
plt.savefig('../assets/diagrams/correlation_causation.svg')
plt.show()

## 5. P-values: What They Mean (and Misconceptions)

- **Correct**: P-value = P(data or more extreme | H₀ true). Small p → inconsistent with H₀.
- **Wrong**: P-value ≠ P(H₀ true). P-value is not probability of null!
- **Wrong**: p=0.05 does not mean 95% sure. It means 5% chance of such extreme data if H₀ were true.

## 6. Capstone: Complete A/B Test Analysis

Load synthetic A/B test data, compute conversion rates, run two-proportion test, report confidence intervals.

In [ ]:
import pandas as pd

df = pd.read_csv('../datasets/sample_data.csv')
print(df.head(10))
print(f"\nVariants: {df['variant'].value_counts()}")

grp = df.groupby('variant').agg({'converted': ['sum', 'count', 'mean']})
grp.columns = ['conversions', 'users', 'rate']
print(grp)

In [ ]:
a = grp.loc['A']
b = grp.loc['B']
z, p = ab_test_two_proportion(int(a['users']), int(a['conversions']), int(b['users']), int(b['conversions']))

print("A/B Test Results")
print(f"  Variant A: {a['conversions']:.0f}/{a['users']:.0f} = {a['rate']:.2%}")
print(f"  Variant B: {b['conversions']:.0f}/{b['users']:.0f} = {b['rate']:.2%}")
print(f"  z = {z:.3f}, p = {p:.4f}")
print(f"  Conclusion: {'Reject H₀ — significant difference' if p < 0.05 else 'Fail to reject — no significant difference'}")

In [ ]:
# Bootstrap CI for difference in conversion rates
def bootstrap_ci_diff(df, variant_col='variant', outcome_col='converted', n_boot=1000, ci=0.95):
    a = df[df[variant_col]=='A'][outcome_col].values
    b = df[df[variant_col]=='B'][outcome_col].values
    diffs = []
    for _ in range(n_boot):
        sa = np.random.choice(a, size=len(a), replace=True)
        sb = np.random.choice(b, size=len(b), replace=True)
        diffs.append(sa.mean() - sb.mean())
    low = np.percentile(diffs, (1-ci)/2 * 100)
    high = np.percentile(diffs, (1+ci)/2 * 100)
    return low, high

low, high = bootstrap_ci_diff(df)
print(f"Bootstrap 95% CI for difference (A - B): [{low:.4f}, {high:.4f}]")

## 7. Summary

- **Hypothesis testing**: z-test, t-test, interpret p-values correctly
- **Confidence intervals**: quantify uncertainty in estimates
- **A/B testing**: two-proportion z-test for conversion/accuracy comparison
- **Correlation ≠ causation**: beware spurious correlations
- **Capstone**: full A/B analysis from data to conclusion

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*